## Customer, Order, Communications History Data Sets - Data Engineering

We will be working with three tables that contains Customer data, Communications data and Order data for a Retail Store. These dataset can be used to understand the consumer behaviour to predict churn, in this example.

These datasets were generated for this demo using a Kaggle dataset below.

Reference: https://www.kaggle.com/uttamp/store-data

In [1]:
import json
import sys
from snowflake.snowpark import Session, Window
from snowflake.snowpark import types as T
from snowflake.snowpark.functions import col
from snowflake.snowpark import functions as F
from snowflake.snowpark.functions import avg, col, max, min, sproc
from snowflake.snowpark.types import *

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()

## Recommended Approach For Preprocessing In Snowflake

#### Use Snowpark DF/Snowpark ML instead of python processing libraries (eg: Pandas)

#### Why? 1. Auto push down compute 2. Distributed Processing.

In [4]:
customers_df = session.table("customers") #Snowpark Dataframe
orders_df = session.table("orders") #Snowpark Dataframe
comm_hist_df = session.table("comm_history") #Snowpark Dataframe

In [5]:
customers_df.show()
orders_df.show()
comm_hist_df.show()

---------------------------------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"CREATED_DT"  |"CITY"   |"STATE"  |"FAV_DELIVERY_DAY"  |"REFILL"  |"DOOR_DELIVERY"  |"PAPERLESS"  |"CUSTOMER_NAME"  |"RETAINED"  |
---------------------------------------------------------------------------------------------------------------------------------------------------
|6H6T6N         |2012-09-28    |Dallas   |TX       |Monday              |0         |0                |0            |flNK83fi4f       |0           |
|APCENR         |2010-12-19    |Dallas   |TX       |Friday              |1         |1                |1            |ljCODBugAV       |1           |
|7UP6MS         |2010-10-03    |Dallas   |TX       |Wednesday           |0         |0                |0            |5B7qnNdi32       |0           |
|7ZEW8G         |2010-10-22    |Houston  |TX       |Thursday            |0         |0                |0         

## Below Code Snippet Is Just To Illustrate Difference Between Pandas & Snowpark Dataframe In Snowflake


In [6]:
# Convert Snowpark DF to pandas
pandas_df = customers_df.to_pandas()  

# Get Snowpark DataFrame size
snowpark_size = sys.getsizeof(customers_df) / (1024*1024)
print(f"Snowpark DataFrame Size (snowpark_df): {snowpark_size:.2f} MB")

# Get pandas DataFrame size
pandas_size = sys.getsizeof(pandas_df) / (1024*1024)
print(f"Pandas DataFrame Size (pandas_df): {pandas_size:.2f} MB")

Snowpark DataFrame Size (snowpark_df): 0.00 MB
Pandas DataFrame Size (pandas_df): 10.57 MB


### Let's apply transformations like joins and aggregations

In [7]:
def createTransformed(dfCust, dfOrd, dfCom):
    
    #Calculate first_order_date, last_order_date and avg_order amount for each customer
    window = Window.partition_by("CUSTOMER_ID")
    df_lastorder = dfOrd.select(col("CUSTOMER_ID"),max("ORDER_DT").over(window).alias("LAST_ORDER_DT")).distinct()
    df_firstorder = dfOrd.select(col("CUSTOMER_ID"),min("ORDER_DT").over(window).alias("FIRST_ORDER_DT")).distinct()
    df_avgorder = dfOrd.select(col("CUSTOMER_ID"),avg("ORDER_AMOUNT").over(window).alias("AVG_ORDER")).distinct()
    df_1 = dfCust.join(dfCom, ["CUSTOMER_ID"])\
                 .join(df_lastorder, ["CUSTOMER_ID"])\
                 .join(df_firstorder, ["CUSTOMER_ID"])\
                 .join(df_avgorder, ["CUSTOMER_ID"])
    
    #calculate DIFF_BETWEEN_LAST_FIRST_DAYS and DIFF_BETWEEN_FIRST_CREATED_DAYS
    df_2 = df_1.with_columns(["DIFF_BETWEEN_LAST_FIRST_DAYS", "DIFF_BETWEEN_FIRST_CREATED_DAYS"], 
                   [F.datediff("DAY", df_1["FIRST_ORDER_DT"].try_cast(DateType()), df_1["LAST_ORDER_DT"].try_cast(DateType())),
                   F.datediff("DAY", df_1["CREATED_DT"].try_cast(DateType()), df_1["FIRST_ORDER_DT"].try_cast(DateType()))
                   ])
    
    return df_2.na.drop()

In [9]:
df=createTransformed(customers_df, orders_df, comm_hist_df)
df.show()

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"CREATED_DT"  |"CITY"   |"STATE"  |"FAV_DELIVERY_DAY"  |"REFILL"  |"DOOR_DELIVERY"  |"PAPERLESS"  |"CUSTOMER_NAME"  |"RETAINED"  |"ESENT"  |"EOPENRATE"  |"ECLICKRATE"  |"LAST_ORDER_DT"  |"FIRST_ORDER_DT"  |"AVG_ORDER"  |"DIFF_BETWEEN_LAST_FIRST_DAYS"  |"DIFF_BETWEEN_FIRST_CREATED_DAYS"  |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|6H6T6N         |2012-09-28    |Dallas   |TX       |Monday              |0     

### Prepare your data for the data scientist - Remove unwanted columns, mask confidential data, do transformations etc and pass it to the DS team with fine grained access control.

In [ ]:
df.write.mode("overwrite").save_as_table('customer_churn')